In [1]:
from lexico import *
from sintactico_ast import *

In [2]:
# Ejemplo de uso 
codigo_fuente = """
int suma(int a, int b) {
  int c = a + b;
  printf(c);
  return c;
  

}
"""

In [3]:
# Anâlisis léxico 
tokens = identificar_tokens(codigo_fuente)
tokens

[('KEYWORD', 'int'),
 ('IDENTIFIER', 'suma'),
 ('DELIMITER', '('),
 ('KEYWORD', 'int'),
 ('IDENTIFIER', 'a'),
 ('DELIMITER', ','),
 ('KEYWORD', 'int'),
 ('IDENTIFIER', 'b'),
 ('DELIMITER', ')'),
 ('DELIMITER', '{'),
 ('KEYWORD', 'int'),
 ('IDENTIFIER', 'c'),
 ('OPERATOR', '='),
 ('IDENTIFIER', 'a'),
 ('OPERATOR', '+'),
 ('IDENTIFIER', 'b'),
 ('DELIMITER', ';'),
 ('KEYWORD', 'printf'),
 ('DELIMITER', '('),
 ('IDENTIFIER', 'c'),
 ('DELIMITER', ')'),
 ('DELIMITER', ';'),
 ('KEYWORD', 'return'),
 ('IDENTIFIER', 'c'),
 ('DELIMITER', ';'),
 ('DELIMITER', '}')]

In [4]:
print('Tokens encontrados')
for tipo, valor in tokens:
    print(f'{tipo}: {valor}')

Tokens encontrados
KEYWORD: int
IDENTIFIER: suma
DELIMITER: (
KEYWORD: int
IDENTIFIER: a
DELIMITER: ,
KEYWORD: int
IDENTIFIER: b
DELIMITER: )
DELIMITER: {
KEYWORD: int
IDENTIFIER: c
OPERATOR: =
IDENTIFIER: a
OPERATOR: +
IDENTIFIER: b
DELIMITER: ;
KEYWORD: printf
DELIMITER: (
IDENTIFIER: c
DELIMITER: )
DELIMITER: ;
KEYWORD: return
IDENTIFIER: c
DELIMITER: ;
DELIMITER: }


In [5]:
import sintactico_ast # Analisis sintactico
try:
    print('\nIniciando análisis sintáctico...')
    parser = Parser(tokens)
    arbol_ast = parser.parsear()
    print('Analisis sintactico completo sin errores!')
except SyntaxError as e:
    print(e)
    


Iniciando análisis sintáctico...
Analisis sintactico completo sin errores!


In [6]:
import json 

In [7]:
def imprimir_ast(nodo):
    if isinstance(nodo, NodoPrograma):
        return {'programa': 'Noname', 
                'funciones': [imprimir_ast(f) for f in nodo.funciones], 
                'main': imprimir_ast(nodo.main)}
    if isinstance(nodo, NodoFuncion):
        return {'Funcion': nodo.nombre[1], 
                'Parametros': [imprimir_ast(p) for p in nodo.parametros],
                'Cuerpo': [imprimir_ast(c) for c in nodo.cuerpo]}
    if isinstance(nodo, NodoParametro):
        return {'Parametro': nodo.nombre[1], 'Tipo': nodo.tipo[1]}
    elif isinstance(nodo, NodoAsignacion):
        return {'Tipo': 'asignacion',
                'variable': nodo.nombre[1], 
                'Expresion': imprimir_ast(nodo.expresion)}
        
    elif isinstance(nodo, NodoOperacion):
        return {'op': nodo.operador[1], 
                'iz': imprimir_ast(nodo.izquierda),
                'der' : imprimir_ast(nodo.derecha)}
    elif isinstance(nodo, NodoRetorno):
        return {'tipo': 'Return',
                'valor': imprimir_ast(nodo.expresion)}
    elif isinstance(nodo, NodoIdentificador):
        return nodo.nombre[1]
    elif isinstance(nodo, NodoNumero):
        return int(nodo.valor[1] if str(nodo.valor[1]).isdigit() else nodo.valor[1]) 

    return None 
        
        

In [8]:
print(json.dumps(imprimir_ast(arbol_ast), indent=1))

{
 "Funcion": "suma",
 "Parametros": [
  {
   "Parametro": "a",
   "Tipo": "int"
  },
  {
   "Parametro": "b",
   "Tipo": "int"
  }
 ],
 "Cuerpo": [
  {
   "Tipo": "asignacion",
   "variable": "c",
   "Expresion": {
    "op": "+",
    "iz": "a",
    "der": "b"
   }
  },
  null,
  {
   "tipo": "Return",
   "valor": "c"
  }
 ]
}


In [9]:
nodoExp = NodoOperacion(NodoNumero(('NUM', 5)), ('OPERATOR', '+'), NodoNumero(('NUM', 8)))
print(json.dumps(imprimir_ast(nodoExp), indent=1))

{
 "op": "+",
 "iz": 5,
 "der": 8
}


In [10]:
print(codigo_fuente)


int suma(int a, int b) {
  int c = a + b;
  printf(c);
  return c;


}



In [11]:
print(arbol_ast.traducirPy())

def suma (a, b):
    c =  a + b
  print(c)
  return c


In [12]:
print(arbol_ast.traducirRuby())

def suma (a, b)
  c =  a + b
  puts c
  return c
 end


In [13]:
# Ejemplo de uso 
codigo_fuente_main = """
int suma(int a, int b) {
  int c = a + b;
  return c;
  
}

int main(){
    int resultado = suma(3, 5);
    return 0;
}
"""

In [14]:
# Anâlisis léxico 
tokens_new = identificar_tokens(codigo_fuente_main)
tokens_new

[('KEYWORD', 'int'),
 ('IDENTIFIER', 'suma'),
 ('DELIMITER', '('),
 ('KEYWORD', 'int'),
 ('IDENTIFIER', 'a'),
 ('DELIMITER', ','),
 ('KEYWORD', 'int'),
 ('IDENTIFIER', 'b'),
 ('DELIMITER', ')'),
 ('DELIMITER', '{'),
 ('KEYWORD', 'int'),
 ('IDENTIFIER', 'c'),
 ('OPERATOR', '='),
 ('IDENTIFIER', 'a'),
 ('OPERATOR', '+'),
 ('IDENTIFIER', 'b'),
 ('DELIMITER', ';'),
 ('KEYWORD', 'return'),
 ('IDENTIFIER', 'c'),
 ('DELIMITER', ';'),
 ('DELIMITER', '}'),
 ('KEYWORD', 'int'),
 ('IDENTIFIER', 'main'),
 ('DELIMITER', '('),
 ('DELIMITER', ')'),
 ('DELIMITER', '{'),
 ('KEYWORD', 'int'),
 ('IDENTIFIER', 'resultado'),
 ('OPERATOR', '='),
 ('IDENTIFIER', 'suma'),
 ('DELIMITER', '('),
 ('NUMBER', '3'),
 ('DELIMITER', ','),
 ('NUMBER', '5'),
 ('DELIMITER', ')'),
 ('DELIMITER', ';'),
 ('KEYWORD', 'return'),
 ('NUMBER', '0'),
 ('DELIMITER', ';'),
 ('DELIMITER', '}')]

In [15]:
 # Analisis sintactico
try:
    print('\nIniciando análisis sintáctico...')
    parser_main = Parser(tokens_new)
    arbol_ast_main = parser_main.parsear()
    print('Analisis sintactico completo sin errores!')
except SyntaxError as e:
    print(e)
    


Iniciando análisis sintáctico...
Analisis sintactico completo sin errores!


In [16]:
print(json.dumps(imprimir_ast(arbol_ast_main), indent=1))

{
 "Funcion": "suma",
 "Parametros": [
  {
   "Parametro": "a",
   "Tipo": "int"
  },
  {
   "Parametro": "b",
   "Tipo": "int"
  }
 ],
 "Cuerpo": [
  {
   "Tipo": "asignacion",
   "variable": "c",
   "Expresion": {
    "op": "+",
    "iz": "a",
    "der": "b"
   }
  },
  {
   "tipo": "Return",
   "valor": "c"
  }
 ]
}
